In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

In [2]:
# 모델 ID (영어 → 한국어 번역)
# model_id = "Helsinki-NLP/opus-mt-tc-big-en-ko"
# model_id = "Helsinki-NLP/opus-mt-en-ko"

# 모델 ID (Meta의 NLLB 600M 모델)
model_id = "facebook/nllb-200-distilled-600M"

In [3]:
# 디바이스 설정 (GPU 사용 가능 시 자동 선택)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# 토크나이저 & 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [5]:
# 번역할 문장들
src_texts = [
    " I watched a movie with my friend yesterday.",
    " Transformers are the backbone of modern NLP.",
    " Deep learning models are changing the job market."
]

In [6]:
# 토큰화
inputs = tokenizer(
        src_texts,
        return_tensors="pt",
        padding=True,
        truncation=True
).to(device)

print(inputs)

{'input_ids': tensor([[256047,    117,  42150,     76,      9,  66481,   2790,   1537,  39011,
         137192, 248075,      2,      1,      1],
        [256047,  16917,   1988,   1044,   2442,    349,  11535,  85900,    452,
          19972,     84, 172993, 248075,      2],
        [256047, 131728, 106668, 141057,   2442, 209683,    349,  11607,  18774,
         248075,      2,      1,      1,      1]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]], device='cuda:0')}


In [8]:
# 번역 생성 (타겟 언어: 한국어 지정)
# 번역 생성 (타겟 언어: 한국어 지정)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("kor_Hang"),
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

최신 버전의 transformers 라이브러리에서 NllbTokenizer의 내부 구조가 변경되면서 발생한 에러입니다. 기존에 타겟 언어의 토큰 ID를 가져올 때 사용하던 lang_code_to_id 속성이 제거되었기 때문에 해당 오류(AttributeError)가 출력되었습니다.

이를 해결하려면 텍스트 형태의 언어 코드를 토큰 ID로 변환해 주는 표준 메서드인 convert_tokens_to_ids를 사용하시면 됩니다.

# ---
1. 파라미터별 상세 설명
inputs

의미: 앞서 토크나이저를 통해 인코딩된 입력 문장 데이터(input_ids, attention_mask 등)를 딕셔너리 형태로 풀어헤쳐() 모델에 전달합니다. 모델이 '무엇을 바탕으로 번역해야 하는지' 알려주는 필수 데이터입니다.

forced_bos_token_id

의미: 생성될 문장의 첫 번째 토큰(BOS: Beginning of Sequence)을 강제로 특정 토큰으로 지정합니다. NLLB 모델 같은 다국어 번역 모델에서는 이 첫 토큰을 타겟 언어 코드(예: 한국어의 경우 kor_Hang)로 지정하여 모델이 어떤 언어로 번역해야 하는지 결정합니다.

max_length

의미: 생성할 최대 토큰 길이를 제한합니다. 여기서는 번역된 한국어 문장이 최대 128토큰을 넘지 않도록 제한하고 있습니다.

num_beams

의미: 텍스트 생성 알고리즘 중 하나인 빔 서치(Beam Search)의 빔 개수를 설정합니다. 값을 4로 지정하면 매 순간 가장 확률이 높은 상위 4개의 문장 후보군을 유지하며 최적의 번역 결과를 찾습니다. (값이 클수록 품질이 좋아질 수 있으나 연산량이 늘어납니다.)

early_stopping

의미: 빔 서치 진행 중, 더 이상 현재 후보군보다 더 나은 문장이 나올 가능성이 없다고 판단되면 설정한 max_length에 도달하기 전이라도 생성을 조기에 종료하여 속도를 높입니다.

In [9]:
# 디코딩
translated = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [10]:
print(type(translated))

<class 'list'>


In [11]:
# 결과 출력
for en, ko in zip(src_texts, translated):
    print(f"[EN] {en}")
    print(f"[KO] {ko}")
    print("-" * 50)

[EN]  I watched a movie with my friend yesterday.
[KO] 어제 친구랑 영화를 봤어요.
--------------------------------------------------
[EN]  Transformers are the backbone of modern NLP.
[KO] 트랜스포머는 현대 NLP의 척추입니다.
--------------------------------------------------
[EN]  Deep learning models are changing the job market.
[KO] 딥러닝 모델은 일자리 시장을 변화시키고 있습니다.
--------------------------------------------------
